# Analytics Interview Practice — **Solutions**
This notebook contains **worked solutions** for the FitPulse case study questions.

> Note: Because dummy data is randomly generated (with a fixed seed), your exact numbers should match if you run from top to bottom.

In [1]:
# --- Setup ---
import numpy as np
import pandas as pd

np.random.seed(42)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

## Generate Dummy Data

In [2]:
# --- Dummy data generation (same as practice notebook) ---

N_USERS = 3000
start = pd.Timestamp("2025-10-01")
end = pd.Timestamp("2025-12-31")
days = (end - start).days + 1

user_id = np.arange(1, N_USERS + 1)

variant = np.random.choice(["control", "treatment"], size=N_USERS, p=[0.5, 0.5])
language = np.random.choice(["en", "hi", "es", "fr"], size=N_USERS, p=[0.55, 0.25, 0.12, 0.08])
country = np.random.choice(["IN", "US", "UK", "ES", "FR"], size=N_USERS, p=[0.55, 0.2, 0.12, 0.07, 0.06])
traffic_source = np.random.choice(["organic", "paid_search", "social", "referral"], size=N_USERS, p=[0.45, 0.3, 0.2, 0.05])

signup_offset = np.random.randint(0, days, size=N_USERS)
signup_date = start + pd.to_timedelta(signup_offset, unit="D")

users = pd.DataFrame({
    "user_id": user_id,
    "signup_date": signup_date,
    "variant": variant,
    "language": language,
    "country": country,
    "traffic_source": traffic_source,
})

# Events: 1–8 sessions per user
sessions_per_user = np.random.randint(1, 9, size=N_USERS)
session_rows = sessions_per_user.sum()

# Expand user_id
event_user_id = np.repeat(user_id, sessions_per_user)

# Session dates after signup (0–30 days)
days_after = np.random.randint(0, 31, size=session_rows)
session_date = (users.set_index("user_id").loc[event_user_id, "signup_date"].values
                + pd.to_timedelta(days_after, unit="D"))

# Minutes spent: treatment slightly higher on average
base_minutes = np.random.gamma(shape=2.2, scale=3.0, size=session_rows)  # positive skew
lift = np.where(users.set_index("user_id").loc[event_user_id, "variant"].values == "treatment", 1.10, 1.00)
minutes_spent = np.clip(base_minutes * lift, 0.2, None)

# Key events flags
src = users.set_index("user_id").loc[event_user_id, "traffic_source"].values
lang = users.set_index("user_id").loc[event_user_id, "language"].values
var = users.set_index("user_id").loc[event_user_id, "variant"].values

p_landing = 0.95
landing_view = (np.random.rand(session_rows) < p_landing).astype(int)

p_signup = 0.18 + (var == "treatment") * 0.02 + (src == "organic") * 0.01
signup_event = (np.random.rand(session_rows) < np.clip(p_signup, 0, 0.6)).astype(int)

p_sub = 0.035 + (var == "treatment") * 0.01 + (src == "paid_search") * 0.008 + (lang == "en") * 0.005
subscribe_event = (np.random.rand(session_rows) < np.clip(p_sub, 0, 0.25)).astype(int)

events = pd.DataFrame({
    "user_id": event_user_id,
    "session_date": pd.to_datetime(session_date),
    "minutes_spent": minutes_spent.round(2),
    "landing_view": landing_view,
    "signup_event": signup_event,
    "subscribe_event": subscribe_event,
})

# Subs: one subscription max per user
sub_users = (events.groupby("user_id")["subscribe_event"].max().reset_index())
sub_users = sub_users[sub_users["subscribe_event"] == 1]["user_id"].values

plan = np.random.choice(["monthly", "annual"], size=len(sub_users), p=[0.82, 0.18])
price = np.where(plan == "monthly", 9.99, 79.99)

first_sub_date = (events[events["subscribe_event"] == 1]
                  .groupby("user_id")["session_date"].min()
                  .reindex(sub_users).values)

churn_flag = (plan == "monthly") & (np.random.rand(len(sub_users)) < 0.28)
churn_days = np.random.randint(30, 91, size=len(sub_users))
churn_date = np.where(churn_flag, pd.to_datetime(first_sub_date) + pd.to_timedelta(churn_days, unit="D"), pd.NaT)

subs = pd.DataFrame({
    "user_id": sub_users,
    "sub_start_date": pd.to_datetime(first_sub_date),
    "plan": plan,
    "price": price,
    "churn_date": pd.to_datetime(churn_date),
})

# Messy values for realism
mask = np.random.rand(N_USERS) < 0.01
users.loc[mask, "language"] = None

mask2 = np.random.rand(len(events)) < 0.002
events.loc[mask2, "minutes_spent"] = events.loc[mask2, "minutes_spent"].astype(str)

print("Rows:", {"users": len(users), "events": len(events), "subs": len(subs)})
display(users.head())
display(events.head())
display(subs.head())

Rows: {'users': 3000, 'events': 13711, 'subs': 560}


C:\Users\abhis\AppData\Local\Temp\ipykernel_39972\705667254.py:95: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['19.39' '11.98' '17.4' '3.09' '3.17' '2.15' '3.91' '2.96' '5.07' '5.34'
 '13.24' '5.68' '4.56' '4.15' '2.79' '3.25' '5.71' '2.44' '4.82' '6.0'
 '3.34' '3.94' '16.1' '10.64' '6.01']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  events.loc[mask2, "minutes_spent"] = events.loc[mask2, "minutes_spent"].astype(str)


,user_id,signup_date,variant,language,country,traffic_source
0,1,2025-12-23,control,hi,US,social
1,2,2025-11-08,treatment,hi,IN,paid_search
2,3,2025-12-06,treatment,en,ES,paid_search
3,4,2025-10-05,treatment,hi,US,organic
4,5,2025-12-04,control,hi,IN,paid_search


,user_id,session_date,minutes_spent,landing_view,signup_event,subscribe_event
0,1,2026-01-01,4.29,1,0,0
1,1,2025-12-31,6.43,1,0,0
2,1,2025-12-28,5.15,1,1,1
3,1,2025-12-23,2.39,1,0,0
4,1,2025-12-27,9.87,1,0,0


,user_id,sub_start_date,plan,price,churn_date
0,1,2025-12-28,monthly,9.99,NaT
1,8,2025-12-15,monthly,9.99,2026-02-16
2,12,2025-12-06,monthly,9.99,NaT
3,14,2025-10-27,monthly,9.99,2026-01-20
4,16,2025-12-24,annual,79.99,NaT


## Q1 — Define KPIs from scratch (solution)
**North Star metric (example):**
- **7-day Paid Conversion per Landing Viewer** = (# users who start a paid subscription within 7 days of their first landing) / (# unique landing viewers)
- **Grain:** weekly (or daily), segmented by variant/source/country as needed.
- **Why:** ties growth (acquisition + activation) directly to monetization while keeping a short feedback loop.

**3 supporting KPIs (leading indicators):**
1) **Landing → Signup rate** = users with signup / users with landing (user-level)
   - *Grain:* cohort week × variant (and by traffic source)
   - *Why:* measures onboarding friction and intent capture.
2) **Signup → Subscribe rate** = users who subscribe / users who signed up (user-level)
   - *Grain:* cohort week × variant
   - *Why:* isolates paywall/pricing/onboarding quality among signups.
3) **D7 Retention** = users active on day 7 after signup / users in cohort
   - *Grain:* signup cohort week × variant
   - *Why:* guards against “buy then churn” and indicates product value.

*(Optional money KPI you’d often add: ARPPU or churn rate for monthly plan.)*

## Q2 — Data model + grain (solution)
- `users` grain: **1 row per user** (`user_id` unique).
- `events` grain: **1 row per user-session** (multiple rows per `user_id`).
- `subs` grain: **0 or 1 row per subscribing user**.

**Join key:** `user_id`.
- Joining `users → events` duplicates user rows (one-to-many), so compute user-level aggregates first (e.g., max flags, user-average minutes) before joining.
- Joining `users → subs` is mostly one-to-one (but still validate uniqueness).

**Data quality checks (examples):**
1) Uniqueness: `users.user_id` unique; `subs.user_id` unique.
2) Range/type checks: `minutes_spent` numeric and non-negative; dates parse correctly; flags are 0/1.

## Q3 — Data cleaning (solution)

In [3]:
# Ensure datetime types
users["signup_date"] = pd.to_datetime(users["signup_date"], errors="coerce")
events["session_date"] = pd.to_datetime(events["session_date"], errors="coerce")

# users.language: fill missing
users["language"] = users["language"].fillna("unknown")

# events.minutes_spent: numeric coercion + fill invalid with overall median
events["minutes_spent"] = pd.to_numeric(events["minutes_spent"], errors="coerce")
median_minutes = events["minutes_spent"].median(skipna=True)
events["minutes_spent"] = events["minutes_spent"].fillna(median_minutes)

users.dtypes, events.dtypes

(user_id                    int32
 signup_date       datetime64[ns]
 variant                   object
 language                  object
 country                   object
 traffic_source            object
 dtype: object,
 user_id                     int32
 session_date       datetime64[ns]
 minutes_spent             float64
 landing_view                int32
 signup_event                int32
 subscribe_event             int32
 dtype: object)

## Q4 — Feature engineering (solution)

In [4]:
# events: event_date
events["event_date"] = events["session_date"].dt.date

# users: Monday-start signup week (week starting Monday)
# Week ending Sunday (W-SUN) => period starts Monday
users["signup_week"] = users["signup_date"].dt.to_period("W-SUN").apply(lambda p: p.start_time)

# events: days_since_signup (needs join)
events = events.merge(users[["user_id", "signup_date"]], on="user_id", how="left")
events["days_since_signup"] = (events["session_date"] - events["signup_date"]).dt.days

events[["user_id", "session_date", "signup_date", "days_since_signup"]].head()

,user_id,session_date,signup_date,days_since_signup
0,1,2026-01-01,2025-12-23,9
1,1,2025-12-31,2025-12-23,8
2,1,2025-12-28,2025-12-23,5
3,1,2025-12-23,2025-12-23,0
4,1,2025-12-27,2025-12-23,4


### Helper: user-level event flags

In [ ]:
user_flags = (events.groupby("user_id")[["landing_view", "signup_event", "subscribe_event"]]
              .max()
              .rename(columns={
                  "landing_view": "has_landing",
                  "signup_event": "has_signup",
                  "subscribe_event": "has_subscribe",
              }))

user_flags = user_flags.merge(users[["user_id", "variant", "traffic_source", "language", "signup_week"]],
                              on="user_id", how="left")

user_flags.head()

## Q5 — Funnel conversion by variant (solution)

In [ ]:
# Base counts per variant
f = user_flags.groupby("variant").agg(
    n_users=("user_id", "nunique"),
    landing_users=("has_landing", "sum"),
    signup_users=("has_signup", "sum"),
    subscribe_users=("has_subscribe", "sum"),
).reset_index()

# Rates
f["landing_rate"] = f["landing_users"] / f["n_users"]
f["signup_rate"] = f["signup_users"] / f["n_users"]
f["subscribe_rate"] = f["subscribe_users"] / f["n_users"]

# Funnel conversions with safe division
f["landing_to_signup"] = np.where(f["landing_users"] > 0, f["signup_users"] / f["landing_users"], np.nan)
f["signup_to_subscribe"] = np.where(f["signup_users"] > 0, f["subscribe_users"] / f["signup_users"], np.nan)
f["landing_to_subscribe"] = np.where(f["landing_users"] > 0, f["subscribe_users"] / f["landing_users"], np.nan)

f[[
    "variant", "n_users", "landing_rate", "signup_rate", "subscribe_rate",
    "landing_to_signup", "signup_to_subscribe", "landing_to_subscribe"
]]

## Q6 — Uplift + interpretation (solution)

In [ ]:
ft = f.set_index("variant")
control = float(ft.loc["control", "landing_to_subscribe"])
treat = float(ft.loc["treatment", "landing_to_subscribe"])

abs_uplift = treat - control
rel_uplift = (treat / control - 1.0) if control != 0 else np.nan

abs_uplift, rel_uplift

**Interpretation (example):**
- Treatment improves landing→subscribe by an **absolute** `abs_uplift` and **relative** `rel_uplift` vs control.
- Before calling this meaningful, validate randomization balance (source/language/country), check novelty effects, and confirm the lift persists across key segments (not driven by one channel).

## Q7 — Inference: minutes spent (solution)

In [ ]:
# We'll compare minutes_spent at the *session* level by variant.
# In real interviews, you can also argue for user-level averages to reduce within-user dependence.

# Attach variant to events
events_v = events.merge(users[["user_id", "variant"]], on="user_id", how="left")

x_c = events_v.loc[events_v["variant"] == "control", "minutes_spent"].astype(float)
x_t = events_v.loc[events_v["variant"] == "treatment", "minutes_spent"].astype(float)

mean_c = x_c.mean()
mean_t = x_t.mean()
diff = mean_t - mean_c

result = None
try:
    from scipy import stats
    tstat, pval = stats.ttest_ind(x_t, x_c, equal_var=False, nan_policy="omit")  # Welch
    result = (tstat, pval)
except Exception as e:
    # Fallback: compute Welch t-stat + approximate p-value using normal approximation (ok for large n)
    import math
    n1, n2 = x_t.notna().sum(), x_c.notna().sum()
    v1, v2 = x_t.var(ddof=1), x_c.var(ddof=1)
    tstat = diff / math.sqrt(v1/n1 + v2/n2)
    # Normal approx p-value
    from math import erf, sqrt
    z = abs(tstat)
    pval = 2 * (1 - 0.5 * (1 + erf(z / sqrt(2))))
    result = (tstat, pval)

mean_c, mean_t, diff, result

**Welch vs Student (answer):**
- Use **Welch’s t-test** by default because the two groups can have **unequal variances** and/or **unequal sample sizes**.
- Student’s t-test assumes equal variances; if you can’t justify that assumption, Welch is safer.

## Q8 — Segmentation: find where treatment wins (solution)

In [ ]:
def segment_landing_to_subscribe(df, segment_col, min_users=200):
    # user-level metric: landing_to_subscribe among landing users
    g = (df.groupby([segment_col, "variant"]).agg(
            n_users=("user_id", "nunique"),
            landing_users=("has_landing", "sum"),
            subscribe_users=("has_subscribe", "sum"),
        ).reset_index())
    g["landing_to_subscribe"] = np.where(g["landing_users"] > 0, g["subscribe_users"] / g["landing_users"], np.nan)

    # filter segments with enough total users across variants
    totals = g.groupby(segment_col)["n_users"].sum()
    keep = totals[totals >= min_users].index
    g = g[g[segment_col].isin(keep)]

    # pivot to compute uplift
    p = g.pivot(index=segment_col, columns="variant", values="landing_to_subscribe")
    p["abs_uplift"] = p.get("treatment") - p.get("control")
    p = p.sort_values("abs_uplift", ascending=False)
    return g.sort_values([segment_col, "variant"]), p

by_source_long, by_source_uplift = segment_landing_to_subscribe(user_flags, "traffic_source", min_users=200)
by_lang_long, by_lang_uplift = segment_landing_to_subscribe(user_flags, "language", min_users=200)

display(by_source_long)
display(by_source_uplift.head(10))

display(by_lang_long)
display(by_lang_uplift.head(10))

print("Top 2 sources by abs uplift:
", by_source_uplift.head(2))
print("
Top 2 languages by abs uplift:
", by_lang_uplift.head(2))

## Q9 — Cohort retention (D1, D7) (solution)

In [ ]:
# Active on a day = any session that day since signup
active_d1 = (events[events["days_since_signup"] == 1]
             .groupby("user_id").size().gt(0).rename("active_d1"))
active_d7 = (events[events["days_since_signup"] == 7]
             .groupby("user_id").size().gt(0).rename("active_d7"))

cohort = users[["user_id", "signup_week", "variant"]].copy()
cohort = cohort.merge(active_d1, on="user_id", how="left").merge(active_d7, on="user_id", how="left")
cohort[["active_d1", "active_d7"]] = cohort[["active_d1", "active_d7"]].fillna(False)

ret = (cohort.groupby(["signup_week", "variant"]).agg(
          cohort_size=("user_id", "nunique"),
          d1_retention=("active_d1", "mean"),
          d7_retention=("active_d7", "mean"),
      ).reset_index()
      .sort_values(["signup_week", "variant"]))

ret.head(20)

## Q10 — Revenue metrics (solution)

In [ ]:
subs_v = subs.merge(users[["user_id", "variant"]], on="user_id", how="left")

total_users = users.groupby("variant")["user_id"].nunique().rename("total_users")
paying_users = subs_v.groupby("variant")["user_id"].nunique().rename("paying_users")
revenue = subs_v.groupby("variant")["price"].sum().rename("total_revenue")

rev = pd.concat([total_users, paying_users, revenue], axis=1).fillna(0)
rev["paying_conversion"] = np.where(rev["total_users"] > 0, rev["paying_users"] / rev["total_users"], np.nan)
rev["ARPU"] = np.where(rev["total_users"] > 0, rev["total_revenue"] / rev["total_users"], np.nan)
rev["ARPPU"] = np.where(rev["paying_users"] > 0, rev["total_revenue"] / rev["paying_users"], np.nan)

# MRR proxy for monthly plan as of cutoff date
cutoff = pd.Timestamp("2026-01-01")
monthly = subs_v[subs_v["plan"] == "monthly"].copy()
monthly["is_active"] = monthly["churn_date"].isna() | (monthly["churn_date"] >= cutoff)

active_monthly = monthly[monthly["is_active"]].groupby("variant")["user_id"].nunique().rename("active_monthly_subs")
rev = rev.merge(active_monthly, left_index=True, right_index=True, how="left").fillna({"active_monthly_subs": 0})
rev["MRR_proxy"] = 9.99 * rev["active_monthly_subs"]

rev.reset_index().rename(columns={"index": "variant"})

## Q11 — Anomaly detection (daily subscribe spikes) (solution)

In [ ]:
# Daily unique subscribers
sub_daily = (events[events["subscribe_event"] == 1]
             .assign(date=lambda d: d["session_date"].dt.date)
             .groupby("date")["user_id"].nunique()
             .rename("subscribing_users"))

# Fill missing dates to avoid biased mean/std
date_index = pd.date_range(events["session_date"].min().normalize(), events["session_date"].max().normalize(), freq="D")
sub_daily = sub_daily.reindex(date_index.date, fill_value=0)

mu = sub_daily.mean()
sigma = sub_daily.std(ddof=1) if sub_daily.std(ddof=1) != 0 else 1.0
z = (sub_daily - mu) / sigma

daily = pd.DataFrame({"date": sub_daily.index, "subscribing_users": sub_daily.values, "z": z.values})
spikes = daily[daily["z"] >= 3].sort_values("z", ascending=False)

spikes, daily.head(10)

## Q12 — 7-day rolling conversion (solution)

In [ ]:
# Daily unique landing viewers
landing_daily = (events[events["landing_view"] == 1]
                 .assign(date=lambda d: d["session_date"].dt.date)
                 .groupby("date")["user_id"].nunique()
                 .rename("landing_users"))

sub_daily2 = (events[events["subscribe_event"] == 1]
              .assign(date=lambda d: d["session_date"].dt.date)
              .groupby("date")["user_id"].nunique()
              .rename("subscribing_users"))

# Full date spine
date_spine = pd.date_range(events["session_date"].min().normalize(), events["session_date"].max().normalize(), freq="D").date

ts = pd.concat([landing_daily, sub_daily2], axis=1).reindex(date_spine).fillna(0)

ts["conversion"] = np.where(ts["landing_users"] > 0, ts["subscribing_users"] / ts["landing_users"], 0.0)

# 7-day rolling mean (no loops)
ts["conversion_7d_roll"] = ts["conversion"].rolling(window=7, min_periods=1).mean()

out = ts.reset_index().rename(columns={"index": "date"})
out.head(15)

---
# Optional “Interview Wrap” (example answers)
1) **Most important insight:** Treatment improves end-to-end conversion, and the gain is concentrated in a subset of acquisition channels/languages (use Q8 results).
2) **Recommendation:** Ship to a larger percentage (or ship with guardrails) if the lift is statistically + practically meaningful and retention doesn’t degrade.
3) **Risks/confounders:** randomization imbalance (traffic source mix), novelty effects, seasonality, session-level dependence, and noisy notebook-like event instrumentation.